# Research next to the platform services

`QuantBook` reads the shared `lean-data` volume (mounted read-only at `/Lean/Data/`, ADR 0004). The platform services are
reached over HTTP with the base URLs injected by `docker-compose.yml`:

* `LEAN_MARKET_DATA_URL` - market-data (`/internal/v1/data/...`: catalog, coverage, history)
* `LEAN_RESULTS_URL` - results (`/internal/v1/results/...`: statistics, charts, orders of finished jobs)

This notebook pulls the same SPY history through `QuantBook` and through the market-data API, then loads a finished
backtest from the results API. Run a backtest first (e.g. `sh deploy/compose/smoke-backtest.sh`) or set `JOB_ID`.

In [ ]:
import json, os
import urllib.parse, urllib.request
import pandas as pd

MARKET_DATA_URL = os.environ.get("LEAN_MARKET_DATA_URL", "http://market-data:8080")
RESULTS_URL = os.environ.get("LEAN_RESULTS_URL", "http://results:8080")
JOB_ID = os.environ.get("JOB_ID")  # optional: a specific finished backtest

def get_json(base, path, **query):
    url = f"{base}{path}" + ("?" + urllib.parse.urlencode(query, doseq=True) if query else "")
    with urllib.request.urlopen(url, timeout=30) as response:
        return json.load(response)

print("market-data:", MARKET_DATA_URL, "| results:", RESULTS_URL, "| data-folder:", Config.get("data-folder"))

## History: QuantBook (in-process, over `lean-data`)

In [ ]:
qb = QuantBook()
spy = qb.add_equity("SPY", Resolution.DAILY, data_normalization_mode=DataNormalizationMode.RAW).symbol
qb_history = qb.history(spy, datetime(2013, 10, 7), datetime(2013, 10, 12), Resolution.DAILY,
                        data_normalization_mode=DataNormalizationMode.RAW)
qb_bars = qb_history.loc[spy][["open", "high", "low", "close", "volume"]]
qb_bars.index = qb_bars.index.date
qb_bars

## History: market-data API (the same files, read by the service)

In [ ]:
bars = get_json(MARKET_DATA_URL, "/internal/v1/data/history", securityType="Equity", market="usa", ticker="SPY",
                resolution="Daily", start="2013-10-07", end="2013-10-11")
api_bars = pd.DataFrame(bars)
# Bar.time is the bar's start in UTC; the date is the trading day
api_bars.index = (pd.to_datetime(api_bars["time"]).dt.tz_convert("America/New_York")).dt.date
api_bars = api_bars[["open", "high", "low", "close", "volume"]].astype(float)
api_bars

In [ ]:
# both are indexed by trading day (QuantBook daily bars end at the 16:00 close); compare the closes
comparison = pd.DataFrame({"quantbook": qb_bars["close"], "market_data_api": api_bars["close"]})
comparison["match"] = (comparison["quantbook"] - comparison["market_data_api"]).abs() < 1e-6
assert comparison["match"].all(), comparison
comparison

## A finished backtest from the results API

In [ ]:
if JOB_ID is None:
    page = get_json(RESULTS_URL, "/internal/v1/results", limit=200)
    finished = [r for r in page["items"] if r.get("status") == "completed"]
    assert finished, "no completed result yet: run a backtest first (sh deploy/compose/smoke-backtest.sh)"
    JOB_ID = finished[-1]["jobId"]
summary = get_json(RESULTS_URL, f"/internal/v1/results/{JOB_ID}")
print("job", JOB_ID, "attempt", summary.get("attempt"), summary.get("status"), summary.get("algorithmId"))
pd.Series(summary.get("statistics") or {}, name="statistics").to_frame()

In [ ]:
charts = get_json(RESULTS_URL, f"/internal/v1/results/{JOB_ID}/charts")
print([c["name"] for c in charts])
equity = get_json(RESULTS_URL, f"/internal/v1/results/{JOB_ID}/charts/" + urllib.parse.quote("Strategy Equity"), maxPoints=500)
series = equity["series"]["Equity"]["values"]
# candlestick points are [time, open, high, low, close]; line points are [time, value]
equity_curve = pd.Series({pd.to_datetime(p[0], unit="s"): p[-1] for p in series}, name="equity")
equity_curve.plot(title=f"Strategy Equity, job {JOB_ID}")
equity_curve.describe()

In [ ]:
orders = get_json(RESULTS_URL, f"/internal/v1/results/{JOB_ID}/orders")
pd.DataFrame(orders["items"])